# E2 OLMoE smoke (Kaggle, GPU T4 x2) - feasibility only, no result

Runs 60 steps with and without the cache to check that OLMoE fits and the cache installs.
Same setup as the series notebook. **Save Version -> Save & Run All (Commit)**; read the log.

In [ ]:
# Feasibility smoke, not part of the protocol.
ARMS = []

In [ ]:
import glob
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.split("\n")
gpus = [g for g in gpus if g.strip()]
print(gpus)
assert len(gpus) == 2 and all("T4" in g for g in gpus), "Set the accelerator to GPU T4 x2"

# Kaggle either keeps the uploaded e2_recipe_kit.zip or unpacks it into a folder.
zips = glob.glob("/kaggle/input/**/e2_recipe_kit.zip", recursive=True)
manifests = glob.glob("/kaggle/input/**/KIT.json", recursive=True)
ROOT = Path("/kaggle/working/soup-rnd")
if ROOT.exists():
    shutil.rmtree(ROOT)
if len(zips) == 1:
    kit_source = zips[0]
    kit_sha = hashlib.sha256(Path(zips[0]).read_bytes()).hexdigest()
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(ROOT)
elif len(manifests) == 1:
    kit_source = str(Path(manifests[0]).parent)
    kit_sha = "KIT.json " + hashlib.sha256(Path(manifests[0]).read_bytes()).hexdigest()
    shutil.copytree(kit_source, ROOT)
else:
    raise AssertionError(f"expected one kit in the inputs, found zips={zips} kits={manifests}")
print("kit", kit_source, kit_sha)
os.chdir(ROOT)
print(subprocess.check_output([sys.executable, "benchmarks/e2/recipe_check/kit.py",
                               "verify", str(ROOT)], text=True))
# Kaggle's image is Python 3.13; Soup caps requires-python below 3.13 only to stop pip
# from pulling untested torch wheels (#358). Keep Kaggle's own torch and allow 3.13.
import torch  # noqa: E402

Path("/tmp/keep_torch.txt").write_text(f"torch=={torch.__version__}\n")
print("python", sys.version.split()[0], "torch", torch.__version__)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-c", "/tmp/keep_torch.txt", "-e", ".[train]"], check=True)
# Kaggle ships torchao 0.10.0; peft refuses any torchao below 0.16 while wrapping the
# model, and no arm quantizes, so it is removed (first W32 start, 2026-11-04).
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)

In [ ]:
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

REVISION = "6d84c48581ece794365f2b8e9cfb043c68ade9c5"
BASE = Path(snapshot_download(
    "allenai/OLMoE-1B-7B-0924", revision=REVISION, cache_dir="/tmp/hf",
    allow_patterns=["*.json", "model-*.safetensors", "tokenizer*"],
    token=UserSecretsClient().get_secret("HF_TOKEN"),
))
assert BASE.name == REVISION, BASE
print(BASE, sorted(p.name for p in BASE.iterdir()))

In [ ]:
stamp = time.strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "e2oc_smoke" / stamp
OUT.mkdir(parents=True)
done = subprocess.run([sys.executable, "benchmarks/e2/olmoe_cache/smoke.py",
                       "--base", str(BASE), "--out", str(OUT)])
print("smoke exit", done.returncode)

In [ ]:
out = Path(f"/kaggle/working/e2oc_smoke_{stamp}.zip")
with zipfile.ZipFile(out, "x", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.rglob("*")):
        if path.is_file() and not path.name.endswith((".safetensors", ".weights.pt", ".bin")):
            z.write(path, Path(stamp) / path.relative_to(OUT))
print(out, out.stat().st_size)